# SmartScan — train the RL scheduler on Kaggle

**SIH 26055 · Smart Scan Strategy for Electronic Warfare**

Trains the from-scratch, action-masked PPO and Double-DQN schedulers, then
evaluates them against the **tuned** sweep baseline on held-out seeds.

### Before you run
1. **Settings → Accelerator → GPU T4 x2** (P100 also fine).
2. **Settings → Persistence → Variables and Files** so a 12 h background run survives.
3. Run all. About 40 minutes for the default budget.

### What this notebook will honestly tell you
At the CPU budgets reachable here, **PPO does not beat the tuned sweep**. It
reaches roughly 90 % of the sweep's return and a *worse* interception ratio,
because it learns to park on one window and abandon the band. That was predicted
in `docs/architecture.md` §17-B before it was measured, and the project's
headline claim rests on the restless-bandit and scan-on-scan policies, which
need no training at all.

The notebook prints the comparison either way. Do not delete the cell that shows
it losing.

In [ ]:
import os, sys, json, time, subprocess
from pathlib import Path

ON_KAGGLE = Path('/kaggle/input').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('./kaggle_working')
WORK.mkdir(parents=True, exist_ok=True)

if ON_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/YOUR_ORG/smartscan.git'], check=False)
try:
    import smartscan
except ImportError:
    sys.path.insert(0, '../..' if not ON_KAGGLE else '/kaggle/input/smartscan-source')
    import smartscan

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'smartscan {smartscan.__version__}, torch {torch.__version__}, device {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Note on where the time goes

The environment is **not** the bottleneck: a step costs ~150 µs because ground
truth is precomputed and stepping is an array slice. The network update is.
That is also why the GPU helps less than you would expect for a model this
small — the batches are tiny and the rollout is sequential.

In [ ]:
import numpy as np
from smartscan.config import load_config
from smartscan.env.gym_env import SmartScanEnv

cfg = load_config('medium.yaml')

# Training seeds are DISJOINT from the evaluation seeds, by construction.
TRAIN_SEEDS = list(range(cfg.run.seed + 1000, cfg.run.seed + 1016))
EVAL_SEEDS  = list(range(cfg.run.seed, cfg.run.seed + 20))
assert not set(TRAIN_SEEDS) & set(EVAL_SEEDS), 'train/eval seed overlap'
print(f'train seeds {TRAIN_SEEDS[0]}..{TRAIN_SEEDS[-1]}   eval seeds {EVAL_SEEDS[0]}..{EVAL_SEEDS[-1]}')

env = SmartScanEnv(cfg, TRAIN_SEEDS[:1])
obs, info = env.reset()
t0 = time.perf_counter()
for _ in range(2000):
    legal = np.flatnonzero(info['action_mask'])
    obs, r, term, _, info = env.step(int(legal[0]))
    if term:
        obs, info = env.reset()
print(f'env step: {(time.perf_counter()-t0)/2000*1e6:.0f} us')
print(f'observation {obs.shape}, {int(info["action_mask"].sum())} of {cfg.n_channels} actions legal')

In [ ]:
from smartscan.agents.rl_agents import train_ppo

PPO_STEPS = 1_500_000   # ~20 min; raise if you have the quota

t0 = time.time()
ppo_net, ppo_log = train_ppo(
    cfg.with_overrides(rl={'ppo': {'entropy_coef': 0.003, 'lr': 5e-4}}),
    TRAIN_SEEDS, total_steps=PPO_STEPS, log_every=20,
)
print(f'PPO trained in {time.time()-t0:.0f}s')
torch.save(ppo_net.state_dict(), WORK / 'ppo_medium.pt')

In [ ]:
from smartscan.agents.rl_agents import train_dqn

DQN_STEPS = 400_000

t0 = time.time()
dqn_net, dqn_log = train_dqn(cfg, TRAIN_SEEDS, total_steps=DQN_STEPS, log_every=20_000)
print(f'DQN trained in {time.time()-t0:.0f}s')
torch.save(dqn_net.state_dict(), WORK / 'dqn_medium.pt')

## Learning curves

Entropy is plotted beside return, because return alone cannot distinguish
"learned a good policy" from "has not learned to decide anything yet". Uniform
over the legal actions is `ln(125) ≈ 4.83`.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for label, log in (('ppo', ppo_log), ('dqn', dqn_log)):
    d = log.as_dict()
    if d['steps']:
        axes[0].plot(d['steps'], d['returns'], marker='o', ms=2, label=label)
        if d['entropy']:
            axes[1].plot(d['steps'], d['entropy'], marker='o', ms=2, label=label)
n_legal = cfg.n_channels - cfg.receiver.ibw_channels + 1
axes[1].axhline(np.log(n_legal), ls='--', color='grey', label=f'uniform over {n_legal}')
axes[0].set_xlabel('environment steps'); axes[0].set_ylabel('episode return'); axes[0].set_title('Return')
axes[1].set_xlabel('environment steps'); axes[1].set_ylabel('entropy / epsilon'); axes[1].set_title('Decisiveness')
for ax in axes: ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig(WORK / 'rl_curves.png', dpi=150); plt.show()

## Evaluate on held-out seeds, against the TUNED sweep

The baseline is the sweep with `dwell_slots` tuned, not the textbook one-slot
version. Beating a deliberately weak incumbent would prove nothing.

In [ ]:
from smartscan.agents import build_agent
from smartscan.agents.rl_agents import DQNScheduler, PPOScheduler
from smartscan.analysis.metrics import evaluate_episode
from smartscan.env.rf_environment import build_episode, generate_scenario
from smartscan.runner import run_episode

ppo_net.cpu().eval(); dqn_net.cpu().eval()
rows = {a: [] for a in ('sequential', 'ucb1', 'whittle', 'phase_locked', 'ppo', 'dqn')}

for seed in EVAL_SEEDS:
    sc = generate_scenario(seed, config=cfg)
    ep = build_episode(sc)
    agents = {
        'sequential':   build_agent('sequential', cfg, seed, sc),
        'ucb1':         build_agent('ucb1', cfg, seed, sc),
        'whittle':      build_agent('whittle', cfg, seed, sc),
        'phase_locked': build_agent('phase_locked', cfg, seed, sc),
        'ppo':          PPOScheduler(cfg, seed, net=ppo_net),
        'dqn':          DQNScheduler(cfg, seed, net=dqn_net),
    }
    for key, agent in agents.items():
        res = run_episode(cfg, seed, agent, scenario=sc, episode=ep)
        rows[key].append(evaluate_episode(res, cfg))

def med(key, metric):
    v = np.asarray([r[metric] for r in rows[key]], float)
    v = v[np.isfinite(v)]
    return float(np.median(v)) if v.size else float('nan')

print(f"{'agent':16}{'TTFI_hard':>11}{'TWIR':>9}{'coverage':>10}{'staleMax':>10}{'reward':>9}")
for key in rows:
    print(f"{key:16}{med(key,'ttfi_hard_median_s'):11.3f}{med(key,'twir_rate'):9.4f}"
          f"{med(key,'coverage'):10.3f}{med(key,'staleness_max_s'):10.3f}"
          f"{med(key,'reward_total'):9.1f}")

In [ ]:
# Paired improvement over the tuned sweep, with confidence intervals.
from smartscan.analysis.metrics import paired_bootstrap_delta

base_twir = np.array([r['twir_rate'] for r in rows['sequential']])
print(f"{'agent':16}{'dTWIR':>10}{'95% CI':>26}{'verdict':>12}")
for key in rows:
    if key == 'sequential':
        continue
    cand = np.array([r['twir_rate'] for r in rows[key]])
    d = paired_bootstrap_delta(cand, base_twir, relative=True, n_boot=4000, seed=cfg.run.seed)
    # twir_rate is higher-is-better, so flip the sign of the time-like delta.
    lo, hi, pt = -d.hi, -d.lo, -d.point
    verdict = 'better' if lo > 0 else ('worse' if hi < 0 else 'inconclusive')
    print(f"{key:16}{100*pt:+9.1f}%  [{100*lo:+7.1f}%, {100*hi:+7.1f}%]{verdict:>12}")

In [ ]:
metrics = {
    'task': 'receiver scheduling (RL)',
    'config_hash': cfg.hash(),
    'ppo_steps': PPO_STEPS,
    'dqn_steps': DQN_STEPS,
    'device': DEVICE,
    'train_seeds': [int(s) for s in TRAIN_SEEDS],
    'eval_seeds': [int(s) for s in EVAL_SEEDS],
    'ppo_log': ppo_log.as_dict(),
    'dqn_log': dqn_log.as_dict(),
    'median_metrics': {
        k: {m: med(k, m) for m in
            ('ttfi_hard_median_s', 'twir_rate', 'coverage', 'staleness_max_s', 'reward_total')}
        for k in rows
    },
    'note': (
        'RL is reported as measured. At this training budget it does not beat the '
        'tuned sweep; the headline claim rests on the restless-bandit and '
        'scan-on-scan policies, which require no training.'
    ),
}
(WORK / 'rl_metrics.json').write_text(json.dumps(metrics, indent=2, default=float))

print('artefacts in', WORK)
for f in sorted(WORK.iterdir()):
    print(f'  {f.name:30} {f.stat().st_size/1024:8.1f} KB')

## Publish

**File → Save Version** persists everything in `/kaggle/working` with the
notebook. To make it reusable as `ew-smart-scan-models`, create a dataset from
the output — or set `PUBLISH = True` in the predictor notebook's final cell,
which writes both sets of artefacts into one dataset.

Inference then needs one line:
```python
import kagglehub
weights = kagglehub.dataset_download('<user>/ew-smart-scan-models')
```